In [1]:
from typing import Annotated, Final, Tuple, Literal, TypedDict
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.store.postgres import PostgresStore
from loadmodel import load_model, load_postgresconfig
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage
from loguru import logger

DB_DSN = load_postgresconfig()
# 初始化模型
model = load_model()

with PostgresStore.from_conn_string(DB_DSN) as store:
    # 1. 初始化数据库，创建长期记忆存储的表
    store.setup()
    # 2. 构建永久记忆数据
    # 2.1 定义永久记忆的命名空间
    USERS_NS: Final[Tuple[str]] = ("users",)
    PREFERENCES_KEY: Final[str] = "preferences"
    # 2.2 定义永久记忆的键值对
    namespace1 = (*USERS_NS, "Alice")
    namespace2 = (*USERS_NS, "Bob")
    namespace3 = (*USERS_NS, "Charlie")

    value1 = {
        "course": "Math",
        "sports": "跑步",
        "food": "鱼香肉丝"
    }
    value2 = {
        "course": "Physics",
        "sports": "篮球",
        "food": "红烧肉"
    }
    value3 = {
        "course": "Chemistry",
        "sports": "游泳",
        "food": "青椒肉丝"
    }

    # 3. 写入永久记忆数据
    store.put(namespace1, PREFERENCES_KEY, value1)
    store.put(namespace2, PREFERENCES_KEY, value2)
    store.put(namespace3, PREFERENCES_KEY, value3)

    for item in store.search(USERS_NS):
        print(item)


Item(namespace=['users', 'Charlie'], key='preferences', value={'food': '青椒肉丝', 'course': 'Chemistry', 'sports': '游泳'}, created_at='2026-08-19T20:45:56.029297+08:00', updated_at='2026-08-19T20:45:56.029297+08:00', score=None)
Item(namespace=['users', 'Bob'], key='preferences', value={'food': '红烧肉', 'course': 'Physics', 'sports': '篮球'}, created_at='2026-08-19T20:45:56.025618+08:00', updated_at='2026-08-19T20:45:56.025618+08:00', score=None)
Item(namespace=['users', 'Alice'], key='preferences', value={'food': '鱼香肉丝', 'course': 'Math', 'sports': '跑步'}, created_at='2026-08-19T20:45:56.018804+08:00', updated_at='2026-08-19T20:45:56.018804+08:00', score=None)
